
# Trabajo Práctico 1

1. Entrene una red de Hopfield '82 con las imágenes binarias disponibles en el campus.

In [ ]:
import os
import numpy as np
import imageio.v2 as imageio
import matplotlib.pyplot as plt

np.random.seed(2837645)

In [ ]:
imgs = []
files = ['paloma.bmp', 'quijote.bmp', 'torero.bmp', 'panda.bmp', 'perro.bmp', 'v.bmp']
for f in files:
  imgs.append(imageio.imread(os.path.join('img', f)))

In [ ]:
for i in range(len(imgs)):
  imgs[i] = 1 - 2*imgs[i]

In [ ]:
fig, axs = plt.subplots(2,3)
for i in range(len(imgs)):
  axs[i//3,i%3].imshow(imgs[i], cmap='Greys', interpolation='nearest')
  axs[i//3,i%3].axis('off')
  axs[i//3,i%3].set_title(files[i])

In [ ]:
def sign(x):
  return 1.0*(x >= 0.0) - 1.0*(x < 0.0)

class RedHopfield:
  def __init__(self, n, eta=1.0):
    self.weights = np.zeros((n,n))
    self.eta = eta
    self.n = n

  def entrenar(self, X):
    delta_weights = np.zeros_like(self.weights)
    #for i in range(self.n):
    #  for j in range(self.n):
    #    if i != j:
    #      delta_weights[i,j] = self.eta * X[i] * X[j]
    delta_weights = self.eta * np.outer(X, X)
    np.fill_diagonal(delta_weights, 0)
    self.weights += delta_weights

  def recuperar(self, X, max_iters=10000):
    X = np.copy(X)
    iters = 0
    while iters < max_iters:
      old_X = X
      for i in np.random.choice(np.arange(self.n), size=self.n, replace=False):
        X[i] = sign(np.sum(self.weights[i,:] * X))
        iters += 1
        if iters >= max_iters:
          break
      if np.all(X == old_X):
        break
    return X

In [ ]:
imgs_train = [imgs[0].flatten(), imgs[1].flatten(), imgs[2].flatten()]
n = imgs_train[0].shape[0]
model = RedHopfield(n)

for i,img in enumerate(imgs_train):
  print(f'Image: {i}')
  model.entrenar(img)

(a) Verifique si la red aprendió las imágenes enseñadas.

In [ ]:
# cuadradas: 50x50, rect: 60x45

fig, axs = plt.subplots(2,3)
for i in range(3):
  axs[0,i].imshow(imgs_train[i].reshape((45,60)), cmap='Greys', interpolation='nearest')
  axs[0,i].set_title('Entrada')
  axs[0,i].axis('off')

  recuperada = model.recuperar(imgs_train[i])
  axs[1,i].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
  axs[1,i].set_title('Recuperada')
  axs[1,i].axis('off')

  assert np.all(imgs_train[i] == recuperada)

In [ ]:
img_test = np.copy(imgs_train[0])
img_test[(45*30):] = -1.0
parcial = model.recuperar(img_test, max_iters=1000)
recuperada = model.recuperar(img_test)
fig, axs = plt.subplots(1,3)
axs[0].imshow(img_test.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[0].set_title('Entrada')
axs[0].axis('off')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

In [ ]:
flip_prob = 0.2

img_test = np.copy(imgs_train[1])
img_test *= np.random.choice([1, -1], p=[1-flip_prob, flip_prob], replace=True, size=img_test.shape)
parcial = model.recuperar(img_test, max_iters=1500)
recuperada = model.recuperar(img_test)
fig, axs = plt.subplots(1,3)
axs[0].imshow(img_test.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[0].axis('off')
axs[0].set_title('Entrada')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

In [ ]:
img_test = np.copy(imgs_train[2])
img_test[(30*30):(45*40):] = 1.0
parcial = model.recuperar(img_test, max_iters=1000)
recuperada = model.recuperar(img_test)
fig, axs = plt.subplots(1,3)
axs[0].imshow(img_test.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[0].set_title('Entrada')
axs[0].axis('off')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

In [ ]:
img_test = np.copy(imgs_train[2])
img_test[0:(45*40):] = 1.0
parcial = model.recuperar(img_test, max_iters=1000)
recuperada = model.recuperar(img_test)
fig, axs = plt.subplots(1,3)
axs[0].imshow(img_test.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[0].set_title('Entrada')
axs[0].axis('off')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

(c) Evalúe la existencia de estados espurios en la red: patrones inversos y combinaciones de un número impar de patrones. (Ver _Spurious States_, en la sección 2.2, Hertz, Krogh & Palmer, pág. 24).

In [ ]:
black = np.ones_like(imgs_train[0])
parcial = model.recuperar(black, max_iters=500)
recuperada = model.recuperar(black)

fig, axs = plt.subplots(1,3)
axs[0].imshow(black.reshape((45,60)), cmap='Greys', interpolation='nearest', vmin=-1, vmax=1)
axs[0].set_title('Entrada')
axs[0].axis('off')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

In [ ]:
img_test = -np.copy(imgs_train[0])
img_test += imgs_train[1]
img_test += imgs_train[2]
img_test = sign(img_test)
parcial = model.recuperar(img_test, max_iters=1000)
recuperada = model.recuperar(img_test)

fig, axs = plt.subplots(1,3)
axs[0].imshow(img_test.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[0].set_title('Entrada')
axs[0].axis('off')
axs[1].imshow(parcial.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[1].axis('off')
axs[1].set_title('Parcial')
axs[2].imshow(recuperada.reshape((45,60)), cmap='Greys', interpolation='nearest')
axs[2].axis('off')
axs[2].set_title('Recuperada')
plt.show()

(d) Realice un entrenamiento con las 6 imágenes disponibles. ¿Es capaz la red de aprender todas las imágenes? Explique.

In [ ]:
imgs_train2 = []
for img in imgs:
  img_50x60 = -np.ones((50, 60))
  h, w = img.shape
  img_50x60[:h,:w] = img
  imgs_train2.append(img_50x60.flatten())

In [ ]:
model2 = RedHopfield(n=imgs_train2[0].shape[0])
for img in imgs_train2:
  model2.entrenar(img)

In [ ]:
fig, axs = plt.subplots(2,6)
fig.set_figheight(5)
fig.set_figwidth(15)
for i,img in enumerate(imgs_train2):
  axs[0,i].imshow(img.reshape((50,60)), cmap='Greys', interpolation='nearest')
  axs[0,i].set_title('Entrada')
  axs[0,i].axis('off')

  recuperada = model2.recuperar(img)
  axs[1,i].imshow(recuperada.reshape((50,60)), cmap='Greys', interpolation='nearest')
  axs[1,i].set_title('Recuperada')
  axs[1,i].axis('off')

  print(f"Imagen {i} {'' if np.all(img == recuperada) else 'no'} recuperada")